In [17]:
# Carrega o dataset e separa as linhas por origem (fonte): reais (conta/cartão), web e sintéticas.
# O teste usa SEMPRE só as reais; web e sintéticas entram apenas no treino.
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from src.limpeza import limpar

df = pd.read_csv("../data/processed/rotulado_v1.csv").dropna(subset=["categoria"])
reais = df[df["fonte"].isin(["conta", "cartao"])].reset_index(drop=True)
web   = df[df["fonte"] == "web"]
sint  = df[df["fonte"] == "sintetico"]
print(f"reais: {len(reais)} | web: {len(web)} | sintéticas: {len(sint)}")

reais: 667 | web: 11 | sintéticas: 66


In [18]:
# Validação cruzada manual: os folds são feitos só com dados reais, e as linhas extras
# (web/sintéticas) são somadas ao treino de cada fold. Retorna o F1 macro (média ± desvio)
# e o F1 médio das categorias pequenas, onde o efeito dos dados extras deve aparecer.
PEQUENAS = ["Assinaturas", "Educação", "Moradia"]

def avaliar(extras, analyzer="word"):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)   
    notas, notas_pequenas = [], []
    for treino_idx, teste_idx in cv.split(reais["descricao"], reais["categoria"]):
        treino = pd.concat([reais.iloc[treino_idx], extras])
        teste  = reais.iloc[teste_idx]

        if analyzer == "char":
            vetorizador = TfidfVectorizer(preprocessor=limpar, analyzer="char_wb", ngram_range=(2, 4))
        else:
            vetorizador = TfidfVectorizer(preprocessor=limpar)
        modelo = make_pipeline(vetorizador, LogisticRegression(max_iter=1000, class_weight="balanced"))

        modelo.fit(treino["descricao"], treino["categoria"])
        previsoes = modelo.predict(teste["descricao"])
        notas.append(f1_score(teste["categoria"], previsoes, average="macro"))
        notas_pequenas.append(f1_score(teste["categoria"], previsoes, labels=PEQUENAS,
                                       average=None, zero_division=0))
    return np.mean(notas), np.std(notas), np.mean(notas_pequenas, axis=0)

In [19]:
# Compara 4 cenários de treino (só real, + web, + sintético, + ambos) nos modelos de
# palavras e de pedaços de letras. O teste é idêntico em todos: a diferença vem só do treino.
cenarios = {
    "só real":                reais.iloc[0:0],          # nenhuma linha extra
    "real + web":             web,
    "real + sintético":       sint,
    "real + web + sintético": pd.concat([web, sint]),
}

resultados = []
for analyzer in ["word", "char"]:
    for nome, extras in cenarios.items():
        media, desvio, pequenas = avaliar(extras, analyzer)
        linha = {"modelo": "palavras" if analyzer == "word" else "letras",
                 "cenário": nome, "extras": len(extras),
                 "F1 macro": f"{media:.3f} ± {desvio:.3f}"}
        linha.update({cat: round(v, 2) for cat, v in zip(PEQUENAS, pequenas)})
        resultados.append(linha)

pd.DataFrame(resultados)

c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificad

,modelo,cenário,extras,F1 macro,Assinaturas,Educação,Moradia
0,palavras,só real,0,0.635 ± 0.066,0.32,0.20,0.42
1,palavras,real + web,11,0.635 ± 0.065,0.32,0.20,0.41
2,palavras,real + sintético,66,0.626 ± 0.074,0.37,0.20,0.41
3,palavras,real + web + sintético,77,0.627 ± 0.075,0.37,0.20,0.41
4,letras,só real,0,0.664 ± 0.088,0.41,0.20,0.44
5,letras,real + web,11,0.646 ± 0.065,0.27,0.20,0.44
6,letras,real + sintético,66,0.646 ± 0.080,0.39,0.17,0.44
7,letras,real + web + sintético,77,0.647 ± 0.082,0.41,0.17,0.44


In [20]:
# Gera a previsão "fora do fold" de cada linha real: cada linha é prevista pelo modelo
# que NÃO a viu no treino. Assim dá para comparar, linha a linha, dois cenários de treino.
def prever_todos(extras, analyzer="char"):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    previsoes = pd.Series(index=reais.index, dtype=object)
    for treino_idx, teste_idx in cv.split(reais["descricao"], reais["categoria"]):
        treino = pd.concat([reais.iloc[treino_idx], extras])
        teste  = reais.iloc[teste_idx]
        if analyzer == "char":
            vetorizador = TfidfVectorizer(preprocessor=limpar, analyzer="char_wb", ngram_range=(2, 4))
        else:
            vetorizador = TfidfVectorizer(preprocessor=limpar)
        modelo = make_pipeline(vetorizador, LogisticRegression(max_iter=1000, class_weight="balanced"))
        modelo.fit(treino["descricao"], treino["categoria"])
        previsoes.iloc[teste_idx] = modelo.predict(teste["descricao"])
    return previsoes

antes  = prever_todos(reais.iloc[0:0])   # só real
depois = prever_todos(sint)              # real + sintético

In [21]:
# Cruza as duas previsões: quais linhas reais o sintético "estragou" e quais ele "consertou".
comparacao = reais[["descricao", "categoria", "banco"]].copy()
comparacao["antes"]  = antes
comparacao["depois"] = depois

pioraram  = comparacao[(comparacao["antes"] == comparacao["categoria"]) & (comparacao["depois"] != comparacao["categoria"])]
melhoraram = comparacao[(comparacao["antes"] != comparacao["categoria"]) & (comparacao["depois"] == comparacao["categoria"])]

print(f"Pioraram: {len(pioraram)} | Melhoraram: {len(melhoraram)}")
print("\nPara onde foram os que pioraram:")
print(pioraram["depois"].value_counts())
pioraram

Pioraram: 7 | Melhoraram: 11

Para onde foram os que pioraram:
depois
Assinaturas    4
Mercado        1
Educação       1
Moradia        1
Name: count, dtype: int64


,descricao,categoria,banco,antes,depois
166,Compra no débito - Zanon's Sbc Comercio V Sao ...,Alimentação,Inter,Alimentação,Mercado
272,Compra no débito - No Estabelecimento Esccomer...,Outros,Inter,Outros,Educação
340,PAGAMENTO ON LINE,Outros,Inter,Outros,Moradia
358,FASHION PHONE,Compras,Itaú,Compras,Assinaturas
379,CHARLOT,Alimentação,Itaú,Alimentação,Assinaturas
407,FONTELEONE BAR,Alimentação,Itaú,Alimentação,Assinaturas
438,AmyrisReal,Compras,Itaú,Compras,Assinaturas


In [22]:
# Para cada palavra dos dados REAIS, guardamos o conjunto de categorias em que ela aparece.
# Um sintético é suspeito se a primeira palavra dele existe nos reais, mas nunca com a mesma categoria.
cats_por_palavra = {}
for texto, categoria in zip(reais["descricao"], reais["categoria"]):
    for palavra in set(limpar(texto).split()):
        cats_por_palavra.setdefault(palavra, set()).add(categoria)

def marca_em_outra_categoria(linha):
    palavras = limpar(linha["descricao"]).split()
    if not palavras:
        return False
    primeira = palavras[0]
    return primeira in cats_por_palavra and linha["categoria"] not in cats_por_palavra[primeira]

sint = sint.copy()   # evita o aviso SettingWithCopyWarning do pandas
sint["regra1_marca"] = sint.apply(marca_em_outra_categoria, axis=1)
sint.loc[sint["regra1_marca"], ["descricao", "categoria", "banco"]]

,descricao,categoria,banco
692,EBANX*DUOLINGO,Assinaturas,Itaú
707,EBANX*UDEMY,Educação,Inter
716,PAG*AULASDEINGLES,Educação,Itaú
717,MP*ESCOLADEMUSICA,Educação,Itaú


In [23]:
sint["regra2_itau"] = (sint["banco"] == "Itaú") & sint["descricao"].str.isupper()
print("Regra 2 marcou:", sint["regra2_itau"].sum(), "linhas")
sint.loc[sint["regra2_itau"], ["descricao", "categoria"]]

Regra 2 marcou: 26 linhas


,descricao,categoria
678,NETFLIX ENTRETENIMENTO,Assinaturas
679,DISNEY PLUS,Assinaturas
680,HBOMAX,Assinaturas
681,GLOBOPLAY,Assinaturas
682,DEEZER BRASIL,Assinaturas
683,APPLE.COM/BILL,Assinaturas
686,KINDLE UNLTD,Assinaturas
687,DROPBOX*2YH7K,Assinaturas
688,ADOBE *CREATIVE CLD,Assinaturas
689,MICROSOFT*365 FAMILY,Assinaturas


In [24]:
pares_reais = set(zip(reais["banco"], reais["categoria"]))
sint["regra3_par"] = [(b, c) not in pares_reais for b, c in zip(sint["banco"], sint["categoria"])]
print("Regra 3 marcou:", sint["regra3_par"].sum(), "linhas")
pd.crosstab(sint["categoria"], sint["banco"])

Regra 3 marcou: 17 linhas


banco,Bradesco,Inter,Itaú,Nubank
categoria,,,,
Assinaturas,5,5,16,0
Educação,2,5,11,4
Moradia,0,11,0,7


In [25]:
sint["ambigua"] = sint["regra1_marca"] | sint["regra2_itau"]           # o que combinamos no chat
sint["ambigua_r3"] = sint["ambigua"] | sint["regra3_par"]              # versão mais rígida
print(sint[["regra1_marca", "regra2_itau", "regra3_par", "ambigua", "ambigua_r3"]].sum())
print("Sobram (regras 1+2):", (~sint["ambigua"]).sum(), "| sobram (1+2+3):", (~sint["ambigua_r3"]).sum())

regra1_marca     4
regra2_itau     26
regra3_par      17
ambigua         27
ambigua_r3      44
dtype: int64
Sobram (regras 1+2): 39 | sobram (1+2+3): 22


In [26]:
COLUNAS = ["descricao", "categoria", "fonte", "banco"]   # tira as colunas de regra antes do treino
cenarios = {
    "só real":                          reais.iloc[0:0],
    "real + todos os sintéticos":       sint[COLUNAS],
    "real + sintéticos filtrados (1+2)": sint.loc[~sint["ambigua"], COLUNAS],
    "real + sintéticos filtrados (1+2+3)": sint.loc[~sint["ambigua_r3"], COLUNAS],
}

resultados = []
for analyzer in ["word", "char"]:
    for nome, extras in cenarios.items():
        media, desvio, pequenas = avaliar(extras, analyzer)
        linha = {"modelo": "palavras" if analyzer == "word" else "letras",
                 "cenário": nome, "extras": len(extras),
                 "F1 macro": f"{media:.3f} ± {desvio:.3f}"}
        linha.update({cat: round(v, 2) for cat, v in zip(PEQUENAS, pequenas)})
        resultados.append(linha)

pd.DataFrame(resultados)

c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificador-transacoes-ptbr\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(
c:\Users\Ricardo Ferreira\Github\classificad

,modelo,cenário,extras,F1 macro,Assinaturas,Educação,Moradia
0,palavras,só real,0,0.635 ± 0.066,0.32,0.20,0.42
1,palavras,real + todos os sintéticos,66,0.626 ± 0.074,0.37,0.20,0.41
2,palavras,real + sintéticos filtrados (1+2),39,0.637 ± 0.068,0.39,0.20,0.41
3,palavras,real + sintéticos filtrados (1+2+3),22,0.629 ± 0.052,0.39,0.20,0.40
4,letras,só real,0,0.664 ± 0.088,0.41,0.20,0.44
5,letras,real + todos os sintéticos,66,0.646 ± 0.080,0.39,0.17,0.44
6,letras,real + sintéticos filtrados (1+2),39,0.658 ± 0.091,0.38,0.20,0.44
7,letras,real + sintéticos filtrados (1+2+3),22,0.644 ± 0.084,0.38,0.20,0.45


In [27]:
sint[COLUNAS + ["regra1_marca", "regra2_itau", "regra3_par"]].to_csv(
    "../data/processed/sinteticos_marcados.csv", index=False)